<a href="https://colab.research.google.com/github/aditya-singh004/MNIST-Digit-Recognization/blob/main/model_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(1212)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
from tensorflow.keras import optimizers

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!unzip digit-recognizer.zip

In [ ]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

In [ ]:
df_train.head()

In [ ]:
print(df_train.shape)
print(df_test.shape)

**TRAIN-TEST SPLIT**

In [ ]:
df_features=df_train.iloc[:,1:785]
df_label=df_train.iloc[:, 0]
x_test=df_test.iloc[:, 0:784]
print(x_test.shape)

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_cv, y_train, y_cv = train_test_split(
    df_features, df_label, test_size = 0.2, random_state = 1212
)
x_train=x_train.values.reshape(-1,784)
x_cv=x_cv.values.reshape(-1,784)
x_test=x_test.values.reshape(-1,784)

**DATA CLEANING, NORMALIZATION AND SELECTION**

In [ ]:
print(x_train[1].min(), x_train[1].max())

In [ ]:
x_train=x_train.astype('float32')   #GOOD TO HAVE IT AS FLOATING POINT NUMBERS
x_cv=x_cv.astype('float32')
x_test=x_test.astype('float32')

x_train /= 255      #NORMALIZATION
x_cv /= 255
x_test /= 255

from tensorflow.keras.utils import to_categorical  #ONE HOT ENCODING
num_digits=10
y_train=to_categorical(y_train, num_digits)
y_cv=to_categorical(y_cv, num_digits)


In [ ]:
print(y_train[0])
print(y_train[5])

**MODEL FITTING**

In [ ]:
ip=Input(shape=(784,))
x=Dense(300, activation='relu')(ip)
x=Dense(100, activation='relu')(x)
x=Dense(100, activation='relu')(x)
x=Dense(200, activation='relu')(x)
op=Dense(10, activation='softmax')(x)


In [ ]:
model=Model(ip, op)
model.summary()

In [ ]:
learning_rate = 0.1
training_epochs = 20
batch_size = 100

sgd = optimizers.SGD(learning_rate=learning_rate)

In [ ]:
model.compile(optimizer='sgd', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
first = model.fit(x_train, y_train,
                     batch_size = batch_size,
                     epochs = training_epochs,
                     verbose = 2,
                     validation_data=(x_cv, y_cv))

In [ ]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

ip=Input(shape=(784,))
x=Dense(300, activation='relu')(ip)
x=Dense(100, activation='relu')(x)
x=Dense(100, activation='relu')(x)
x=Dense(200, activation='relu')(x)
op=Dense(10, activation='softmax')(x)

# Adam optimizer
adam = Adam(learning_rate=learning_rate)

model2 = Model(ip, op)

model2.compile(loss='categorical_crossentropy',
               optimizer=adam,
               metrics=['accuracy'])

In [ ]:
second = model2.fit(x_train, y_train,
                      batch_size = batch_size,
                      epochs = training_epochs,
                      verbose = 2,
                      validation_data=(x_cv, y_cv))

In [ ]:
learning_rate = 0.01
training_epochs = 20
batch_size = 100

In [ ]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

ip=Input(shape=(784,))
x=Dense(300, activation='relu')(ip)
x = Dropout(0.3)(x)

x=Dense(100, activation='relu')(x)
x = Dropout(0.3)(x)

x=Dense(100, activation='relu')(x)
x = Dropout(0.3)(x)

x=Dense(200, activation='relu')(x)
op=Dense(10, activation='softmax')(x)

# Adam optimizer
adam = Adam(learning_rate=learning_rate)

model3 = Model(ip, op)

model3.compile(loss='categorical_crossentropy',
               optimizer=adam,
               metrics=['accuracy'])

In [ ]:
history = model3.fit(x_train, y_train,
                    batch_size = batch_size,
                    epochs = training_epochs,
                    validation_data=(x_cv, y_cv))

In [ ]:
test_pred = pd.DataFrame(model3.predict(x_test, batch_size=200))
test_pred = pd.DataFrame(test_pred.idxmax(axis = 1))
test_pred.index.name = 'ImageId'
test_pred = test_pred.rename(columns = {0: 'Label'}).reset_index()
test_pred['ImageId'] = test_pred['ImageId'] + 1

test_pred.head()

In [ ]:
test_pred.to_csv('mnist_submission.csv', index = False)


In [ ]:
x_train = x_train.reshape(-1,28,28,1)
x_cv = x_cv.reshape(-1,28,28,1)
x_test = x_test.reshape(-1,28,28,1)

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.models import Model

In [ ]:
ip = Input(shape=(28,28,1))

# First convolution layer
x = Conv2D(32, (3,3), activation='relu', padding='same')(ip)
x = BatchNormalization()(x)
x = Conv2D(32, (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = Dropout(0.25)(x)

# Second convolution layer
x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2))(x)
x = Dropout(0.25)(x)

x = Flatten()(x)

# NN
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

op = Dense(10, activation='softmax')(x)

cnn_model = Model(ip, op)

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
cnn_model.fit(
    x_train, y_train,
    batch_size=128,
    epochs=15,
    validation_data=(x_cv, y_cv)
)